# Chapter 4 Notebook: 5G Traffic Forecasting, Calibration & Replay Setup
Run cells **1 through 7 in order** from the project root (`5G_Project_sim`). All outputs are saved to `data/` and `results/`.

In [ ]:
# Cell 1: Ensure project root working directory & import path
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if 'src' not in sys.path:
    sys.path.insert(0, 'src')
print('Working directory:', os.getcwd())

In [ ]:
# Cell 2: Log environment versions & hardware profile to results/logs/runtime_env.json
import json, platform
import numpy as np, pandas as pd
os.makedirs('results/logs', exist_ok=True)
env_info = {
    'python_version': sys.version.split()[0],
    'os_platform': f'{platform.system()} {platform.release()}',
    'packages': {'numpy': np.__version__, 'pandas': pd.__version__}
}
with open('results/logs/runtime_env.json', 'w') as f:
    json.dump(env_info, f, indent=4)
print('Saved results/logs/runtime_env.json:', env_info)

In [ ]:
# Cell 3: Generate synthetic 5G telemetry and clean into one slice table
seed = 42
rng = np.random.default_rng(seed)
n_steps = 2400
ts = pd.date_range('2026-01-01 00:00:00', periods=n_steps, freq='5min')
t = np.arange(n_steps, dtype=float)
steps_per_day = 288.0
diurnal = 0.5 + 0.5 * np.sin(2.0 * np.pi * (t % steps_per_day) / steps_per_day - np.pi / 2.0)
wknd_factor = np.where(ts.dayofweek >= 5, 0.82, 1.05)
ar = np.zeros(n_steps)
eps = rng.normal(0.0, 12.0, size=n_steps)
for i in range(1, n_steps):
    ar[i] = 0.65 * ar[i - 1] + eps[i]
bursts = np.where(rng.random(n_steps) < 0.025, rng.uniform(60.0, 180.0, size=n_steps), 0.0)
raw_df = pd.DataFrame({
    'timestamp': ts,
    'eMBB_mbps': np.round(np.clip(260.0 * diurnal * wknd_factor + ar * 1.4 + bursts, 30.0, 750.0), 2),
    'mMTC_mbps': np.round(np.clip(95.0 + 25.0 * np.cos(2.0 * np.pi * t / steps_per_day) + rng.normal(0.0, 8.0, size=n_steps), 15.0, 220.0), 2),
    'URLLC_mbps': np.round(np.clip(55.0 + 15.0 * diurnal + rng.normal(0.0, 6.0, size=n_steps) + np.where(bursts > 0, 25.0, 0.0), 10.0, 180.0), 2),
})
raw_df['total_demand_mbps'] = np.round(raw_df[['eMBB_mbps', 'mMTC_mbps', 'URLLC_mbps']].sum(axis=1), 2)
os.makedirs('data/raw', exist_ok=True)
raw_df.to_csv('data/raw/traffic_telemetry.csv', index=False)
clean_df = raw_df.sort_values('timestamp').drop_duplicates('timestamp').reset_index(drop=True)
for col in ['eMBB_mbps', 'mMTC_mbps', 'URLLC_mbps']:
    cap = float(np.percentile(clean_df[col].values, 99.5))
    clean_df[col] = np.clip(clean_df[col].values, 0.0, cap)
clean_df['total_demand_mbps'] = clean_df[['eMBB_mbps', 'mMTC_mbps', 'URLLC_mbps']].sum(axis=1)
os.makedirs('data/processed', exist_ok=True)
clean_df.to_csv('data/processed/clean_slice_table.csv', index=False)
clean_df.head(3)

In [ ]:
# Cell 4: Build causal features & chronological 70/15/15 split with ordering check
lookback, roll_win, horizon = 15, 12, 5
s = clean_df['total_demand_mbps'].values.astype(float)
hrs = clean_df['timestamp'].dt.hour.values + clean_df['timestamp'].dt.minute.values / 60.0
wknd = (clean_df['timestamp'].dt.dayofweek.values >= 5).astype(float)
X_rows, y_vals, t_vals = [], [], []
for idx in range(max(lookback, roll_win), len(s) - horizon):
    lags = s[idx - lookback + 1 : idx + 1][::-1]
    w = s[idx - roll_win + 1 : idx + 1]
    X_rows.append(np.concatenate([lags, [np.mean(w), np.std(w), np.sin(2*np.pi*hrs[idx]/24), np.cos(2*np.pi*hrs[idx]/24), wknd[idx]]]))
    y_vals.append(s[idx + horizon])
    t_vals.append(clean_df['timestamp'].iloc[idx])
cols = [f'lag_{k}' for k in range(1, lookback + 1)] + ['roll_mean', 'roll_std', 'sin_tod', 'cos_tod', 'weekend']
X = pd.DataFrame(X_rows, columns=cols)
y = np.array(y_vals)
t_ser = pd.Series(t_vals)
n = len(X)
i_tr, i_ca = int(0.70 * n), int(0.70 * n) + int(0.15 * n)
X_tr, y_tr, t_tr = X.iloc[:i_tr], y[:i_tr], t_ser.iloc[:i_tr]
X_ca, y_ca, t_ca = X.iloc[i_tr:i_ca], y[i_tr:i_ca], t_ser.iloc[i_tr:i_ca]
X_te, y_te, t_te = X.iloc[i_ca:], y[i_ca:], t_ser.iloc[i_ca:]
assert t_tr.max() < t_ca.min() and t_ca.max() < t_te.min()
print('Splits -> Train:', len(X_tr), 'Cal:', len(X_ca), 'Test:', len(X_te))

In [ ]:
# Cell 5: Train Random Forest, run sanity checks, and save calibration_summary.csv
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
rf = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=seed, n_jobs=-1).fit(X_tr, y_tr)
pred_te = rf.predict(X_te)
shuf_pred = RandomForestRegressor(n_estimators=25, max_depth=6, random_state=seed, n_jobs=-1).fit(X_tr, rng.permutation(y_tr)).predict(X_te)
print('Last-value MAE:', round(mean_absolute_error(y_te, X_te['lag_1']), 3))
print('RF Test MAE:', round(mean_absolute_error(y_te, pred_te), 3), '| R2:', round(r2_score(y_te, pred_te), 4))
print('Shuffled target MAE:', round(mean_absolute_error(y_te, shuf_pred), 3))
cal_res = y_ca - rf.predict(X_ca)
os.makedirs('results/tables', exist_ok=True)
cal_df = pd.DataFrame([
    {'target': t_cov, 'q_mbps': round(float(np.quantile(cal_res, t_cov)), 3), 'emp_cov': round(float(np.mean(y_te <= (pred_te + float(np.quantile(cal_res, t_cov))))), 4)}
    for t_cov in [0.80, 0.90, 0.95]
])
cal_df.to_csv('results/tables/calibration_summary.csv', index=False)
cal_df

In [ ]:
# Cell 6: Replay simulation under capacity C with URLLC-first priority -> simulation_summary.json
cap_C = 1000.0
q90 = float(cal_df.loc[cal_df['target'] == 0.90, 'q_mbps'].iloc[0])
tail = clean_df.iloc[-len(y_te):].reset_index(drop=True)
tot = np.maximum(tail['total_demand_mbps'].values, 1e-6)
u_sh, e_sh, m_sh = tail['URLLC_mbps'].values / tot, tail['eMBB_mbps'].values / tot, tail['mMTC_mbps'].values / tot
alloc = np.clip(pred_te + q90, 0.0, cap_C)
unmet, waste, u_short = [], [], []
for k in range(len(y_te)):
    u_s = min(alloc[k] * u_sh[k], cap_C)
    rem = max(0.0, cap_C - u_s)
    other_req = alloc[k] * (e_sh[k] + m_sh[k])
    e_s = rem * (alloc[k] * e_sh[k] / other_req) if other_req > 0 else 0.0
    m_s = rem * (alloc[k] * m_sh[k] / other_req) if other_req > 0 else 0.0
    srv = u_s + e_s + m_s
    unmet.append(max(0.0, float(y_te[k]) - srv))
    waste.append(max(0.0, float(alloc[k]) - float(y_te[k])))
    u_short.append(max(0.0, float(y_te[k] * u_sh[k]) - u_s))
sim_out = {
    'capacity_C_mbps': cap_C,
    'mean_unmet_mbps': round(float(np.mean(unmet)), 3),
    'sla_violation_pct': round(float(np.mean(np.array(unmet) > 0) * 100.0), 2),
    'mean_waste_mbps': round(float(np.mean(waste)), 3),
    'mean_urllc_shortfall_mbps': round(float(np.mean(u_short)), 3),
}
with open('results/tables/simulation_summary.json', 'w') as f:
    json.dump(sim_out, f, indent=4)
sim_out

In [ ]:
# Cell 7: Confirm all artifacts exist
for p in ['data/raw/traffic_telemetry.csv', 'results/logs/runtime_env.json', 'results/tables/calibration_summary.csv', 'results/tables/simulation_summary.json']:
    print(p, '->', os.path.exists(p))